# GraphRAG Explorer

This notebook walks through the full GraphRAG workflow interactively.

**Prerequisites:**
1. Activate the venv: `source venv/Scripts/activate`
2. Install Jupyter kernel: `python -m ipykernel install --user --name graphrag-playground`
3. Copy `.env.example` → `.env` and set your `GRAPHRAG_API_KEY`
4. Select the `graphrag-playground` kernel in Jupyter

**What GraphRAG does:**
- Reads your text documents
- Uses an LLM to extract **entities** (people, places, concepts) and **relationships**
- Clusters them into **communities** using graph algorithms (Leiden)
- Generates LLM **summaries** of each community
- At query time, uses this structure to provide richer, more grounded answers than naive RAG

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Load .env
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env')
    print('Loaded .env')
except ImportError:
    print('python-dotenv not installed — set GRAPHRAG_API_KEY manually')

ROOT = Path('..').resolve()
RAGTEST = ROOT / 'ragtest'
print(f'ROOT: {ROOT}')
print(f'API key set: {bool(os.getenv("GRAPHRAG_API_KEY"))}')

## Step 1 — Initialise the project

In [ ]:
# Copy sample texts to ragtest/input/
input_dir = RAGTEST / 'input'
input_dir.mkdir(parents=True, exist_ok=True)

for f in (ROOT / 'data' / 'sample_texts').glob('*.txt'):
    dest = input_dir / f.name
    dest.write_text(f.read_text('utf-8'), 'utf-8')
    print(f'  copied {f.name}')

# Init (safe to re-run — won't overwrite existing settings)
result = subprocess.run(
    [sys.executable, '-m', 'graphrag', 'init', '--root', str(RAGTEST)],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

In [ ]:
# Inspect the generated settings file
settings_file = RAGTEST / 'settings.yaml'
if settings_file.exists():
    print(settings_file.read_text()[:3000])
else:
    print('settings.yaml not found — re-run the init cell')

## Step 2 — Run the indexer

> ⚠️ This calls your LLM for every chunk and entity — **it costs money and takes several minutes**.
> With the 3 sample documents (~3000 words each), expect ~$0.50-2.00 depending on your model.
>
> You can monitor progress in the cell output.

In [ ]:
# Run the indexer — streams output live
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, '-m', 'graphrag', 'index', '--root', str(RAGTEST)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(f'\nIndexing finished with exit code {proc.returncode}')

## Step 3 — Inspect the knowledge graph artefacts

In [ ]:
import pandas as pd
from pathlib import Path

OUTPUT = RAGTEST / 'output'

def load(keyword):
    files = sorted(OUTPUT.rglob(f'*{keyword}*.parquet'), key=lambda p: p.stat().st_mtime)
    if not files:
        print(f'No parquet file found for: {keyword}')
        return pd.DataFrame()
    print(f'Loading {files[-1].relative_to(ROOT)}')
    return pd.read_parquet(files[-1])

In [ ]:
entities = load('entities')
print(f'Columns: {list(entities.columns)}')
entities[['title','type','description']].head(20)

In [ ]:
# Top entities by degree (most connected)
if 'degree' in entities.columns:
    entities[['title','type','degree']].sort_values('degree', ascending=False).head(15)
else:
    entities[['title','type']].head(15)

In [ ]:
rels = load('relationships')
print(f'Total relationships: {len(rels)}')
rels[['source','target','description']].head(20)

In [ ]:
reports = load('community_reports')
print(f'Communities: {len(reports)}')
if not reports.empty:
    display(reports[['title','level','summary']].head(10))
    # Print a full report
    print('\n--- Full report sample ---')
    col = 'full_content' if 'full_content' in reports.columns else 'summary'
    print(reports.iloc[0][col])

## Step 4 — Visualise the graph (optional)

Install extras: `pip install networkx matplotlib`

In [ ]:
try:
    import networkx as nx
    import matplotlib.pyplot as plt

    G = nx.Graph()
    if not entities.empty:
        for _, row in entities.head(30).iterrows():
            G.add_node(row['title'], node_type=row.get('type', 'UNKNOWN'))

    if not rels.empty:
        for _, row in rels.head(60).iterrows():
            if row['source'] in G.nodes and row['target'] in G.nodes:
                G.add_edge(row['source'], row['target'])

    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, seed=42, k=2)
    nx.draw_networkx(G, pos, node_size=600, font_size=7, node_color='steelblue',
                     font_color='white', edge_color='#aaa', width=0.8)
    plt.title('GraphRAG Knowledge Graph (top 30 entities)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Install networkx and matplotlib to visualise: pip install networkx matplotlib')

## Step 5 — Query

### Global search — broad, thematic questions

In [ ]:
def query(method, question):
    result = subprocess.run(
        [sys.executable, '-m', 'graphrag', 'query',
         '--root', str(RAGTEST),
         '--method', method,
         '--query', question],
        capture_output=True, text=True
    )
    print(result.stdout or result.stderr)

query('global', 'What are the main themes across all the documents?')

In [ ]:
query('global', 'Who are the most influential figures described in these texts and why?')

### Local search — entity-focused questions

In [ ]:
query('local', 'What was Julius Caesar\'s relationship with Pompey and Crassus?')

In [ ]:
query('local', 'What did Geoffrey Hinton, Yann LeCun, and Yoshua Bengio contribute to AI?')

In [ ]:
query('local', 'How does ocean acidification relate to climate change?')

## Step 6 — Try your own documents

1. Add `.txt` files to `data/sample_texts/` (or a new folder)
2. Copy them to `ragtest/input/`
3. Re-run the indexer cell
4. Query away!

GraphRAG works especially well on:
- Long documents with many named entities (reports, books, papers)
- Corpora where understanding relationships between concepts matters
- Questions that require synthesising information across many documents